# Datacenter Load Profile Generator

Creates hourly load profiles for datacenters based on real data from Airon.

**Data source:** Airon effektprofiler (2025) — real PUE and power data for liquid-cooled hyperscale datacenters.

**Key characteristics:**
- Baseload: ~90% load factor, 24/7/365 operations
- Counter-seasonal: Peak in summer (cooling), flat in winter
- No weekly variation: Operates identically all days
- Liquid-cooled: Much lower cooling overhead than air-cooled

**Approach:**
1. Extract daily power data from Airon Excel (Hyperscale liquid-cooled, 100% utilization)
2. Derive monthly multipliers from daily total power (IT + cooling)
3. Compensate fall cooling drop-off by mirroring spring ramp-up
4. Recalibrate summer hourly pattern amplitude from PUE data
5. Output same format as before for downstream compatibility

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import json
import calendar
from pathlib import Path
from collections import defaultdict
import openpyxl
import matplotlib.pyplot as plt

# Configuration
YEAR = 2025
IS_LEAP_YEAR = (YEAR % 4 == 0 and YEAR % 100 != 0) or (YEAR % 400 == 0)
HOURS_IN_YEAR = 8784 if IS_LEAP_YEAR else 8760

# Input/output paths
DATA_PATH = Path('.').resolve()  # .../datacenters/
INPUT_XLSX = DATA_PATH / 'Airon effektprofiler.xlsx'
OUTPUT_DIR = DATA_PATH.parent    # .../load_profiles/
OUTPUT_CSV = OUTPUT_DIR / f'profile_datacenters_{YEAR}.csv'
OUTPUT_JSON = OUTPUT_DIR / 'profile_datacenters_patterns.json'

# Excel row numbers (1-indexed) for Hyperscale liquid-cooled section
ROW_HYPERSCALE_PUE_LC = 26    # "Vätskekyld" — liquid-cooled PUE per day
ROW_HYPERSCALE_IT = 27        # "IT-Last" — constant IT load (0.72 MW per 1MW reserved)
ROW_HYPERSCALE_TOTAL = 28     # "IT-last + kyla" — total power incl. cooling

print(f"Year: {YEAR} ({'leap year' if IS_LEAP_YEAR else 'regular year'})")
print(f"Hours in year: {HOURS_IN_YEAR}")
print(f"Input: {INPUT_XLSX}")
print(f"Output CSV: {OUTPUT_CSV}")
print(f"Output JSON: {OUTPUT_JSON}")

## 2. Parse Airon Excel Data

Extract daily power data for Hyperscale liquid-cooled datacenter (100% utilization)
and daily PUE values from the PUE25 sheet.

In [ ]:
wb = openpyxl.load_workbook(INPUT_XLSX, data_only=True)

# --- DC sheet: Hyperscale liquid-cooled daily data ---
ws_dc = wb['DC']
n_days = 365  # 2025 is not a leap year

# Extract 365 daily values for key rows (columns B through NB = cols 2..366)
daily_pue_lc = [ws_dc.cell(ROW_HYPERSCALE_PUE_LC, c).value for c in range(2, 2 + n_days)]
daily_it_load = [ws_dc.cell(ROW_HYPERSCALE_IT, c).value for c in range(2, 2 + n_days)]
daily_total = [ws_dc.cell(ROW_HYPERSCALE_TOTAL, c).value for c in range(2, 2 + n_days)]

print("Hyperscale liquid-cooled (per 1 MW reserved, 90% usable, 100% utilized):")
print(f"  IT load:     constant = {daily_it_load[0]:.4f} MW")
print(f"  PUE (LC):    min={min(daily_pue_lc):.4f}  max={max(daily_pue_lc):.4f}")
print(f"  Total power: min={min(daily_total):.4f}  max={max(daily_total):.4f}")

# --- PUE25 sheet: daily PUE values (air-cooled baseline) ---
ws_pue = wb['PUE25']
pue25_dates = []
pue25_values = []
for row in range(1, ws_pue.max_row + 1):
    dt = ws_pue.cell(row, 1).value
    pue = ws_pue.cell(row, 2).value
    if dt and pue:
        pue25_dates.append(dt)
        pue25_values.append(pue)

print(f"\nPUE25 sheet: {len(pue25_values)} daily values")
print(f"  Date range: {pue25_dates[0].date()} to {pue25_dates[-1].date()}")
print(f"  PUE range:  {min(pue25_values):.4f} to {max(pue25_values):.4f}")

wb.close()

## 3. Derive Monthly Multipliers

Group daily total power by month, then compensate for fall cooling anomaly.

In [ ]:
# Group daily total power by month
monthly_raw = defaultdict(list)
day_idx = 0
for month in range(1, 13):
    days_in_month = calendar.monthrange(YEAR, month)[1]
    for d in range(days_in_month):
        if day_idx < len(daily_total):
            monthly_raw[month].append(daily_total[day_idx])
            day_idx += 1

# Also group PUE25 by month for cross-validation
pue_monthly = defaultdict(list)
for dt, pue in zip(pue25_dates, pue25_values):
    pue_monthly[dt.month].append(pue)

# Raw monthly averages
print("Raw monthly averages (total power per 1MW reserved):")
raw_monthly_avg = {}
for m in range(1, 13):
    raw_monthly_avg[m] = sum(monthly_raw[m]) / len(monthly_raw[m])
    pue_avg = sum(pue_monthly[m]) / len(pue_monthly[m]) if pue_monthly[m] else 0
    print(f"  Month {m:2d}: total={raw_monthly_avg[m]:.4f}  PUE25={pue_avg:.4f}")

In [ ]:
# Compensate fall cooling drop-off by mirroring spring ramp-up
#
# Airon warned: "kylning baseras på våra verkliga värden, här ska den
# återgå till baseline efter sommaren, så ni kan behöva kompensera för det"
#
# Strategy: July is the peak month. Mirror the spring ramp (Jan→Jul) onto
# fall (Jul→Dec) to create a symmetric seasonal curve.
# Spring: months 1-6 ramp up to Jul peak
# Fall:   month 8 mirrors Jun, month 9 mirrors May, etc.

PEAK_MONTH = 7  # July

# Build compensated monthly averages
compensated_monthly = {}
for m in range(1, 13):
    if m <= PEAK_MONTH:
        # Spring side: use raw data as-is
        compensated_monthly[m] = raw_monthly_avg[m]
    else:
        # Fall side: mirror from spring
        # Month 8 mirrors month 6, month 9 mirrors month 5, etc.
        mirror_month = PEAK_MONTH - (m - PEAK_MONTH)
        if mirror_month >= 1:
            compensated_monthly[m] = raw_monthly_avg[mirror_month]
        else:
            # For months far from peak, use Jan baseline
            compensated_monthly[m] = raw_monthly_avg[1]

# Normalize to sum = 12.0
total = sum(compensated_monthly.values())
MONTHLY_MULT = {m: v / total * 12.0 for m, v in compensated_monthly.items()}

print("Compensated monthly multipliers (sum=12.0):")
print(f"{'Month':>7} {'Raw':>8} {'Compensated':>13} {'Multiplier':>12}")
for m in range(1, 13):
    print(f"  {m:2d}     {raw_monthly_avg[m]:.4f}    {compensated_monthly[m]:.4f}       {MONTHLY_MULT[m]:.4f}")

print(f"\nSum: {sum(MONTHLY_MULT.values()):.4f}")
print(f"Peak/min ratio: {max(MONTHLY_MULT.values())/min(MONTHLY_MULT.values()):.2f}x")

## 4. Determine Summer/Winter Month Boundaries

Use PUE25 monthly averages to find which months have significant cooling overhead.

In [ ]:
# Determine summer months from compensated monthly data
# We can't use PUE25 raw data directly because the fall anomaly inflates Sep-Dec.
# Instead, use the compensated monthly multipliers: months significantly above
# the winter baseline count as "summer" (active cooling months).
winter_baseline_mult = min(MONTHLY_MULT.values())  # Jan or Feb
# Summer = months where multiplier is > 2% above baseline
summer_threshold_mult = winter_baseline_mult * 1.02

SUMMER_MONTHS = [m for m in range(1, 13) if MONTHLY_MULT[m] > summer_threshold_mult]
WINTER_MONTHS = [m for m in range(1, 13) if m not in SUMMER_MONTHS]

print(f"Winter baseline multiplier: {winter_baseline_mult:.4f}")
print(f"Summer threshold (baseline × 1.02): {summer_threshold_mult:.4f}")
print(f"\nMonthly multiplier vs threshold:")
for m in range(1, 13):
    marker = "SUMMER" if m in SUMMER_MONTHS else "winter"
    print(f"  Month {m:2d}: mult={MONTHLY_MULT[m]:.4f}  [{marker}]")
print(f"\nSummer months: {SUMMER_MONTHS}")
print(f"Winter months: {WINTER_MONTHS}")

## 5. Derive Hourly Patterns

Winter: flat (no cooling variation). Summer: bell curve calibrated to liquid-cooled PUE swing.

In [ ]:
# Winter hourly pattern: completely flat (free cooling, no variation)
HOURLY_WINTER = {h: 1.00 for h in range(24)}

# Summer hourly pattern: temperature-following bell curve
# Calibrate amplitude from liquid-cooled PUE data.
#
# For liquid-cooled DCs, IT load is constant. Only cooling varies with temperature.
# The hourly swing comes from cooling power following outdoor temperature.
#
# Use PUE data from reliable spring/summer months (Apr-Jul) to estimate
# the intra-day thermal swing, avoiding the fall anomaly in the raw data.
calibration_months = [4, 5, 6, 7]  # Spring/summer PUE is reliable
summer_pue_values = []
for m in calibration_months:
    summer_pue_values.extend(pue_monthly[m])

# The day-to-day PUE range in summer approximates the thermal swing
pue_summer_max = np.percentile(summer_pue_values, 95)  # hot day
pue_summer_min = np.percentile(summer_pue_values, 5)   # cool day

# For liquid-cooled: convert air PUE to LC PUE range
# DC sheet row 4 shows conversion: LC PUE = 1.1 + (air_PUE - 1.2) * (0.15/0.25)
# This maps air PUE range [1.2, 1.45] to LC PUE range [1.1, 1.25]
def air_to_lc_pue(air_pue):
    return 1.1 + (air_pue - 1.2) * (0.15 / 0.25)

lc_pue_max = air_to_lc_pue(pue_summer_max)
lc_pue_min = air_to_lc_pue(pue_summer_min)

# Peak/trough ratio for total power = PUE_max / PUE_min (IT load cancels out)
hourly_ratio = lc_pue_max / lc_pue_min
print(f"Calibration months: {calibration_months}")
print(f"Summer PUE (air):    p5={pue_summer_min:.4f}  p95={pue_summer_max:.4f}")
print(f"Summer PUE (LC):     p5={lc_pue_min:.4f}  p95={lc_pue_max:.4f}")
print(f"Estimated hourly peak/min ratio: {hourly_ratio:.3f}x")

# Build bell curve shape (peak at 13:00, min at 01:00)
# Use a cosine-based shape centered on hour 13
bell_raw = {}
for h in range(24):
    phase = 2 * np.pi * ((h - 13) % 24) / 24
    bell_raw[h] = np.cos(phase)  # range [-1, 1]

# Scale to desired peak/min ratio
min_val = 1.0
max_val = hourly_ratio
HOURLY_SUMMER = {}
for h in range(24):
    HOURLY_SUMMER[h] = min_val + (bell_raw[h] + 1) / 2 * (max_val - min_val)

# Normalize both patterns to sum = 24.0
winter_sum = sum(HOURLY_WINTER.values())
summer_sum = sum(HOURLY_SUMMER.values())
HOURLY_WINTER = {h: v / winter_sum * 24 for h, v in HOURLY_WINTER.items()}
HOURLY_SUMMER = {h: v / summer_sum * 24 for h, v in HOURLY_SUMMER.items()}

print(f"\nWinter hourly: flat at {HOURLY_WINTER[0]:.4f}")
print(f"Summer hourly: min={min(HOURLY_SUMMER.values()):.4f} (h={min(HOURLY_SUMMER, key=HOURLY_SUMMER.get)})"
      f"  max={max(HOURLY_SUMMER.values()):.4f} (h={max(HOURLY_SUMMER, key=HOURLY_SUMMER.get)})")
print(f"Summer peak/min: {max(HOURLY_SUMMER.values())/min(HOURLY_SUMMER.values()):.3f}x")

In [ ]:
# Weekday multipliers: no variation (24/7/365 operation)
WEEKDAY_MULT = {d: 1.00 for d in range(7)}
print("Weekday multipliers: all 1.00 (no weekly variation)")

## 6. Generate Hourly Profile

In [ ]:
# Generate timestamp index for the year
timestamps = pd.date_range(
    start=f'{YEAR}-01-01 00:00:00',
    end=f'{YEAR}-12-31 23:00:00',
    freq='h'
)
print(f"Generated {len(timestamps)} timestamps ({HOURS_IN_YEAR} expected)")

def get_hourly_multiplier(month: int, hour: int) -> float:
    if month in SUMMER_MONTHS:
        return HOURLY_SUMMER[hour]
    return HOURLY_WINTER[hour]

# Generate raw profile values
values = []
for ts in timestamps:
    hourly_mult = get_hourly_multiplier(ts.month, ts.hour)
    monthly_mult = MONTHLY_MULT[ts.month]
    weekday_mult = WEEKDAY_MULT[ts.dayofweek]
    values.append(hourly_mult * monthly_mult * weekday_mult)

df = pd.DataFrame({'timestamp': timestamps, 'raw_value': values})

# Normalize so values sum to 1.0
total = df['raw_value'].sum()
df['value'] = df['raw_value'] / total

print(f"\nNormalized profile:")
print(f"  Sum:  {df['value'].sum():.10f}")
print(f"  Min:  {df['value'].min():.10f}")
print(f"  Max:  {df['value'].max():.10f}")
print(f"  Mean: {df['value'].mean():.10f}")

## 7. Validate

In [ ]:
print("=== VALIDATION ===")
df['month'] = df['timestamp'].dt.month
df['hour'] = df['timestamp'].dt.hour

# 1. Sum equals 1.0
total_sum = df['value'].sum()
check1 = abs(total_sum - 1.0) < 1e-9
print(f"1. Sum equals 1.0: {'PASS' if check1 else 'FAIL'} (sum = {total_sum:.10f})")

# 2. All values positive
check2 = (df['value'] > 0).all()
print(f"2. All values positive: {'PASS' if check2 else 'FAIL'}")

# 3. July > January (seasonal pattern from real data)
july_avg = df[df['month'] == 7]['value'].mean()
jan_avg = df[df['month'] == 1]['value'].mean()
seasonal_ratio = july_avg / jan_avg
check3 = seasonal_ratio > 1.0
print(f"3. July > January: {'PASS' if check3 else 'FAIL'} (ratio: {seasonal_ratio:.2f}x)")

# 4. Summer hourly variation exists but is modest (liquid-cooled)
summer_df = df[df['month'].isin(SUMMER_MONTHS)]
hour_peak = summer_df.groupby('hour')['value'].mean().max()
hour_trough = summer_df.groupby('hour')['value'].mean().min()
summer_hourly_ratio = hour_peak / hour_trough
check4 = 1.0 < summer_hourly_ratio < 2.0  # LC should be modest, not huge
print(f"4. Summer hourly ratio modest: {'PASS' if check4 else 'FAIL'} ({summer_hourly_ratio:.3f}x, expect 1.1-1.3x)")

# 5. Winter is flat
winter_df = df[df['month'].isin(WINTER_MONTHS)]
winter_by_hour = winter_df.groupby('hour')['value'].mean()
winter_ratio = winter_by_hour.max() / winter_by_hour.min()
check5 = winter_ratio < 1.001
print(f"5. Winter is flat: {'PASS' if check5 else 'FAIL'} (ratio: {winter_ratio:.4f})")

# 6. No weekday variation
df['weekday'] = df['timestamp'].dt.dayofweek
weekday_avg = df.groupby('weekday')['value'].mean()
weekday_ratio = weekday_avg.max() / weekday_avg.min()
check6 = abs(weekday_ratio - 1.0) < 0.01
print(f"6. No weekday variation: {'PASS' if check6 else 'FAIL'} (ratio: {weekday_ratio:.4f})")

all_passed = all([check1, check2, check3, check4, check5, check6])
print(f"\n{'ALL CHECKS PASSED' if all_passed else 'SOME CHECKS FAILED'}")

## 8. Visualize

In [ ]:
# Annual profile - daily totals
df['date'] = df['timestamp'].dt.date
daily_sum = df.groupby('date')['value'].sum()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_sum.index, daily_sum.values, linewidth=0.5, alpha=0.8)
ax.fill_between(daily_sum.index, daily_sum.values, alpha=0.3)
ax.set_xlabel('Date')
ax.set_ylabel('Daily Load (fraction of annual)')
ax.set_title(f'Datacenter Load Profile — {YEAR} (Airon Hyperscale LC)')
ax.grid(True, alpha=0.3)

# Mark summer period
summer_start = pd.Timestamp(f'{YEAR}-{min(SUMMER_MONTHS):02d}-01')
summer_end = pd.Timestamp(f'{YEAR}-{max(SUMMER_MONTHS):02d}-28')
ax.axvspan(summer_start, summer_end, alpha=0.1, color='red', label='Summer (active cooling)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Monthly multipliers: raw vs compensated
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Left: Raw vs compensated monthly averages
ax = axes[0]
raw_vals = [raw_monthly_avg[m] for m in range(1, 13)]
comp_vals = [compensated_monthly[m] for m in range(1, 13)]
x = np.arange(12)
ax.bar(x - 0.2, raw_vals, 0.35, label='Raw (Airon)', alpha=0.7, color='tab:blue')
ax.bar(x + 0.2, comp_vals, 0.35, label='Compensated (mirrored fall)', alpha=0.7, color='tab:orange')
ax.set_xticks(x)
ax.set_xticklabels(months, rotation=45)
ax.set_ylabel('Total Power (MW per 1MW reserved)')
ax.set_title('Monthly Power: Raw vs Compensated')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Right: Final multipliers
ax = axes[1]
mult_vals = [MONTHLY_MULT[m] for m in range(1, 13)]
colors = ['red' if m in SUMMER_MONTHS else 'blue' for m in range(1, 13)]
bars = ax.bar(months, mult_vals, color=colors, alpha=0.7)
ax.axhline(y=1.0, color='black', linestyle='--', alpha=0.5)
ax.set_ylabel('Monthly Multiplier')
ax.set_title('Final Monthly Multipliers (sum=12)')
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, mult_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.2f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# Hourly patterns: winter vs summer
fig, ax = plt.subplots(figsize=(12, 5))
winter_vals = [HOURLY_WINTER[h] for h in range(24)]
summer_vals = [HOURLY_SUMMER[h] for h in range(24)]

ax.plot(range(24), winter_vals, marker='o', label='Winter (flat)', color='blue')
ax.plot(range(24), summer_vals, marker='s', label='Summer (LC bell curve)', color='red')
ax.axhline(y=1.0, color='black', linestyle='--', alpha=0.5)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Hourly Multiplier')
ax.set_title(f'Hourly Patterns: Winter vs Summer (LC peak/min = {max(HOURLY_SUMMER.values())/min(HOURLY_SUMMER.values()):.2f}x)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

## 9. Export

In [ ]:
# Prepare and save CSV
df_export = pd.DataFrame({
    'hour': range(len(df)),
    'value': df['value'].values
})
df_export.to_csv(OUTPUT_CSV, index=False)

print(f"Saved CSV: {OUTPUT_CSV}")
print(f"  Rows: {len(df_export)}, Sum: {df_export['value'].sum():.10f}")
print(f"  Size: {OUTPUT_CSV.stat().st_size / 1024:.1f} KB")

In [ ]:
# Prepare and save JSON patterns
patterns = {
    'hourly_winter': [HOURLY_WINTER[h] for h in range(24)],
    'hourly_summer': [HOURLY_SUMMER[h] for h in range(24)],
    'weekday': [WEEKDAY_MULT[d] for d in range(7)],
    'monthly': [MONTHLY_MULT[m] for m in range(1, 13)],
    'summer_months': SUMMER_MONTHS,
    'winter_months': WINTER_MONTHS,
    'source_year': YEAR,
    'description': 'Datacenter load profile derived from Airon real data (Hyperscale liquid-cooled)',
    'notes': {
        'hourly_winter': 'Flat pattern — free cooling months have no hourly variation',
        'hourly_summer': 'Cosine bell curve calibrated to LC PUE swing (modest amplitude)',
        'weekday': 'All 1.0 — no weekly variation (24/7 operation)',
        'monthly': 'Derived from Airon daily total power, fall mirrored from spring',
        'summer_peak_hour': 13,
        'summer_peak_min_ratio': round(max(HOURLY_SUMMER.values()) / min(HOURLY_SUMMER.values()), 3),
        'seasonal_ratio': round(MONTHLY_MULT[7] / MONTHLY_MULT[1], 3),
        'fall_compensation': 'Months 8-12 mirrored from months 6-2 (spring ramp) to compensate Airon fall anomaly',
    },
    'data_source': 'Airon effektprofiler (2025) — Hyperscale liquid-cooled, 100% utilization, 1MW blocks'
}

with open(OUTPUT_JSON, 'w') as f:
    json.dump(patterns, f, indent=2)

print(f"Saved JSON: {OUTPUT_JSON}")
print(f"  Size: {OUTPUT_JSON.stat().st_size / 1024:.1f} KB")

In [ ]:
# Summary
print("=" * 60)
print("DATACENTER PROFILE GENERATION COMPLETE")
print("=" * 60)
print(f"\nSource: Airon Hyperscale liquid-cooled (2025)")
print(f"Profile year: {YEAR} ({HOURS_IN_YEAR} hours)")
print(f"\n1. CSV: {OUTPUT_CSV}")
print(f"   {len(df_export)} hourly values, normalized sum=1.0")
print(f"\n2. JSON: {OUTPUT_JSON}")
print(f"   Summer months: {SUMMER_MONTHS}")
print(f"   Winter months: {WINTER_MONTHS}")
print(f"\n3. Key characteristics:")
print(f"   Seasonal ratio (Jul/Jan): {MONTHLY_MULT[7]/MONTHLY_MULT[1]:.3f}x")
print(f"   Summer hourly peak/min:   {max(HOURLY_SUMMER.values())/min(HOURLY_SUMMER.values()):.3f}x")
print(f"   Winter hourly variation:  flat (1.000)")
print(f"   Weekday variation:        none (1.000)")
print(f"\n4. Fall compensation applied: months 8-12 mirrored from spring")